# AdAHuman — patch optimization on Colab

Optimizes the person-suppression patch (RQ1b) on a GPU. Everything else in
the artifact runs locally on CPU.

**What crosses the boundary.** Only the source bundle goes up, and only the
patch tensor plus its training log and run record come back. Colab never sees
the held-out evaluation pool: `train_patch` is entitled to read `attack_dev`
and `reference` only, and `PoolDataset` refuses to construct against anything
else. The patch is therefore a frozen input to local evaluation, and the
training device never enters a reported measurement.

**Before running:** Runtime → Change runtime type → GPU.

This notebook uses no interactive upload or download widgets, so it behaves
the same in the Colab web UI and in a VS Code notebook client. Files move via
the filesystem: put the bundle somewhere the next cell searches, and collect
outputs from the directory the last cell reports.

## 1. Confirm the GPU

In [4]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > GPU.'
print('torch', torch.__version__, '|', torch.cuda.get_device_name(0))

Thu Aug  6 04:04:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Locate and extract the source bundle

Generate it locally first:

```bash
bash scripts/make_colab_bundle.sh
```

Then put `adahuman_colab_src.tar.gz` where this cell can find it — any of:

- **Google Drive** (recommended): drop it at the top level of *My Drive*, or
  in a `AdAHuman/` folder there. Set `MOUNT_DRIVE = True` below.
- **Colab session storage**: `/content/`, via the Files pane.
- **Local working directory**, if you are running this against a local
  runtime from VS Code.

The cell prints the bundle's sha256. Record it — it identifies the exact code
that produced the patch, and the training run log is matched against it.

In [9]:
import hashlib, os, pathlib, tarfile

BUNDLE_NAME = 'adahuman_colab_src.tar.gz'
WORKDIR = pathlib.Path('/content/adahuman')
MOUNT_DRIVE = False   # set False if not using Drive

if MOUNT_DRIVE and not pathlib.Path('/content/drive/MyDrive').is_dir():
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as exc:
        print(f'Drive not mounted ({exc}); searching other locations')

SEARCH_DIRS = [
    pathlib.Path("./"), 
    pathlib.Path.cwd(),
    pathlib.Path('/content'),
    pathlib.Path('/content/drive/MyDrive'),
    pathlib.Path('/content/drive/MyDrive/AdAHuman'),
    pathlib.Path.home(),
]

def find_bundle():
    """First exact match wins; otherwise fall back to a shallow glob."""
    for directory in SEARCH_DIRS:
        candidate = directory / BUNDLE_NAME
        if candidate.is_file():
            return candidate
    for directory in SEARCH_DIRS:
        if not directory.is_dir():
            continue
        matches = sorted(directory.glob(f'**/{BUNDLE_NAME}'))
        if matches:
            return matches[0]
    return None

bundle = find_bundle()
if bundle is None:
    searched = '\n  '.join(str(d) for d in SEARCH_DIRS)
    raise FileNotFoundError(
        f'{BUNDLE_NAME} not found. Searched:\n  {searched}\n\n'
        f'Build it locally with `bash scripts/make_colab_bundle.sh`, then copy '
        f'it into Google Drive (top level of My Drive) or /content, and rerun '
        f'this cell. To use a path not listed above, append it to SEARCH_DIRS.'
    )

digest = hashlib.sha256(bundle.read_bytes()).hexdigest()
print(f'bundle  {bundle}')
print(f'size    {bundle.stat().st_size / 1024:.0f} KiB')
print(f'sha256  {digest}')

WORKDIR.mkdir(parents=True, exist_ok=True)
with tarfile.open(bundle) as archive:
    archive.extractall(WORKDIR)
os.chdir(WORKDIR)

print(f'\nextracted to {WORKDIR}')
print(sorted(p.name for p in WORKDIR.iterdir()))

FileNotFoundError: adahuman_colab_src.tar.gz not found. Searched:
  .
  /content/adahuman
  /content
  /content/drive/MyDrive
  /content/drive/MyDrive/AdAHuman
  /root

Build it locally with `bash scripts/make_colab_bundle.sh`, then copy it into Google Drive (top level of My Drive) or /content, and rerun this cell. To use a path not listed above, append it to SEARCH_DIRS.

## 3. Dependencies

Colab ships its own torch build, which is kept: reinstalling the pinned CPU
versions here would discard CUDA support. The version actually used is
captured in the run log, so the difference from the local pin is recorded
rather than hidden.

In [ ]:
!pip install -q pycocotools scikit-learn PyYAML 2>&1 | tail -2
import torch, torchvision
print('torch      ', torch.__version__)
print('torchvision', torchvision.__version__)

## 4. Fetch COCO val2017

In [ ]:
!bash scripts/00_fetch_coco.sh

## 5. Verify the pools match the frozen manifests

The manifests came up in the bundle. This re-derives pool membership from the
seed and checks it against them, confirming that the Colab environment selects
the same images the local freeze did.

In [ ]:
!python scripts/02_freeze_pools.py

## 6. Timing probe

Five steps, then an estimate of what each epoch budget costs. Use it to pick
the epoch count below. Writes nothing.

In [ ]:
!python scripts/04_train_patch.py --probe-timing --workers 2

## 7. Train

Watch `max-person`: it starts near 0.83 and should fall. If it plateaus well
above the 0.5 decision threshold, the attack is weak — report that rather than
extending the run, and note it in `LIMITATIONS.md`. Extending until the number
looks good is how a schedule turns into a result.

`--write-steps` freezes `attack.steps` in the protocol on completion.

In [ ]:
!python scripts/04_train_patch.py --epochs 30 --workers 2 --write-steps

## 8. Collect the outputs

Copies the results to Drive when it is mounted, so they survive the session
and can be retrieved without a download widget. Falls back to leaving them in
the session and printing their paths.

Place `patch_v1.*` in `artifacts/` and the run log in `logs/` in the local
repo, and replace `configs/protocol_v1.yaml` with the copy returned here —
`--write-steps` modified it. The run log is what ties the patch to the code,
the seed, and the GPU that produced it.

In [ ]:
import glob, hashlib, pathlib, shutil

outputs = (
    sorted(glob.glob('artifacts/patch_v1*'))
    + sorted(glob.glob('logs/*train_patch.json'))
    + ['configs/protocol_v1.yaml']
)
outputs = [p for p in outputs if pathlib.Path(p).is_file()]
if not outputs:
    raise FileNotFoundError('no outputs found; did the training cell finish?')

drive_root = pathlib.Path('/content/drive/MyDrive')
destination = drive_root / 'AdAHuman_results' if drive_root.is_dir() else None
if destination:
    destination.mkdir(parents=True, exist_ok=True)

print(f'{"sha256":18s}  {"size":>9s}  file')
for path in outputs:
    source = pathlib.Path(path)
    digest = hashlib.sha256(source.read_bytes()).hexdigest()
    print(f'{digest[:16]}    {source.stat().st_size:9,d}  {path}')
    if destination:
        shutil.copy2(source, destination / source.name)

if destination:
    print(f'\ncopied {len(outputs)} files to {destination}')
    print('Retrieve them from Google Drive, then commit them in the local repo.')
else:
    print(f'\nDrive is not mounted. Files remain in {pathlib.Path.cwd()}.')
    print('Download them via the Files pane, or set MOUNT_DRIVE = True in cell 2')
    print('and rerun this cell.')

## 9. Preview the patch

A sanity check, not a result. A patch that is uniform or saturated usually
means the optimizer diverged.

In [ ]:
from IPython.display import Image, display
display(Image('artifacts/patch_v1.png'))